# Isentropic streamfunction

Computes the isentropic streamfunction $\Psi(z, \theta_e)$ (Pauluis and Mrowiec 2013) from 3D model output:

1. compute $\theta_e$ at every grid point with `isentropic.thermo`,
2. bin the vertical mass flux $\rho w$ in $(z, \theta_e)$ with `isentropic.iso_multi_integral`, level by level,
3. sum the conditional sums over all files and times (= time average),
4. $\Psi(z, \theta_e)$ = cumulative sum over $\theta_e$ of the mass flux, divided by the domain area.

Use the **Python (iso)** kernel.

## Configuration
Edit the variable names to match the files (see the `ncdump -h` output).

In [ ]:
from pathlib import Path

DATA_DIR = Path("/scratch/ttg2015/mailbox/OPaul/2026-10-23_vddgrid_3d_30Days_Perpetual")
PATTERN = "*.nc"

# SAM 6.11 3D snapshot output
#   W (m/s, already at scalar levels), TABS (K), QV, QN (g/kg), p(z) (mb), PP (Pa)
#   QN = cloud liquid + ice, partitioned with SAM's temperature ramp between 253.16 and 273.16 K
VAR = dict(w="W", T="TABS", p="p", pp="PP", qv="QV", qn="QN")
Z_DIM = "z"
TIME_DIM = "time"
T_ICE, T_LIQ = 253.16, 273.16   # SAM's partition of QN into ice and liquid

# theta_e bins (K)
TH_MIN, TH_MAX, NBIN = 300.0, 370.0, 141

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import isentropic as iso
from isentropic import thermo

files = sorted(DATA_DIR.glob(PATTERN))
print(f"{len(files)} files")
ds0 = xr.open_dataset(files[0])
ds0

## Thermodynamic fields at one time

In [ ]:
def load_time(ds, it):
    """Return rho*w and theta_e (nz, ny, nx) and the density profile (nz) at time index it."""
    snap = ds.isel({TIME_DIM: it})
    get = lambda name: snap[name].values.astype(float)
    T = get(VAR["T"])
    qv = np.maximum(get(VAR["qv"]) * 1e-3, 1e-10)           # g/kg -> kg/kg
    qn = np.maximum(get(VAR["qn"]) * 1e-3, 0.0)
    liq_frac = np.clip((T - T_ICE) / (T_LIQ - T_ICE), 0.0, 1.0)
    ql, qi = qn * liq_frac, qn * (1.0 - liq_frac)
    p = get(VAR["p"])[:, None, None] * 100.0 + get(VAR["pp"])   # mb -> Pa, plus perturbation
    the = thermo.equivalent_potential_temperature(p, T, qv, ql, qi)
    rho = thermo.density(p, T, qv, ql, qi).mean((1, 2))        # density profile (anelastic)
    return rho[:, None, None] * get(VAR["w"]), the, rho


rhow, the, rho = load_time(ds0, 0)
print("theta_e range:", np.nanmin(the), np.nanmax(the))
print("rho*w range:", np.nanmin(rhow), np.nanmax(rhow))

## Conditional sums over all files and times

In [ ]:
z = ds0[Z_DIM].values
nz = len(z)
MF = np.zeros((nz, NBIN))
AREA = np.zeros((nz, NBIN))
ATHE = np.zeros((nz, NBIN))
ntimes = 0

for f in files:
    with xr.open_dataset(f) as ds:
        for it in range(ds.sizes[TIME_DIM]):
            rhow, the, rho = load_time(ds, it)
            mask = ~np.isfinite(the)
            out, area = iso.iso_multi_integral(np.stack([rhow, the]), the, mask, 1.0,
                                               th_min=TH_MIN, th_max=TH_MAX, nbin=NBIN)
            MF += out[0]
            ATHE += out[1]
            AREA += area
            ntimes += 1
            if it % 24 == 0:
                print(f.name, "time", it)

print(ntimes, "snapshots")

## Streamfunction

In [ ]:
dth = (TH_MAX - TH_MIN) / (NBIN - 1)
theta = TH_MIN + dth * np.arange(NBIN)
psi = iso.streamfunction(MF, AREA)                 # kg m-2 s-1, at theta + dth/2
the_mean = ATHE.sum(-1) / AREA.sum(-1)             # horizontal-mean theta_e profile

fig, ax = plt.subplots(figsize=(7, 5))
lim = np.abs(psi).max()
pc = ax.pcolormesh(theta + dth / 2, z / 1e3, psi, cmap="RdBu_r", vmin=-lim, vmax=lim, shading="auto")
ax.contour(theta + dth / 2, z / 1e3, psi, levels=np.linspace(-lim, lim, 20), colors="k", linewidths=0.4)
ax.plot(the_mean, z / 1e3, "k", lw=2, label=r"mean $\theta_e$")
ax.set_xlabel(r"$\theta_e$ (K)")
ax.set_ylabel("z (km)")
ax.set_title(f"Isentropic streamfunction ({ntimes} snapshots)")
ax.legend(loc="upper right")
fig.colorbar(pc, label=r"$\Psi$ (kg m$^{-2}$ s$^{-1}$)")
plt.show()

## Two-stream mass fluxes

In [ ]:
ts = iso.two_stream(MF, AREA)
fig, ax = plt.subplots(figsize=(4, 5))
ax.plot(ts["M_up"], z / 1e3, label=r"$M^+$")
ax.plot(-ts["M_down"], z / 1e3, label=r"$-M^-$")
ax.set_xlabel(r"mass flux (kg m$^{-2}$ s$^{-1}$)")
ax.set_ylabel("z (km)")
ax.legend()
plt.show()

## Save the conditional sums

In [ ]:
out = xr.Dataset(
    {"MF": (("z", "theta_e"), MF), "AREA": (("z", "theta_e"), AREA), "ATHE": (("z", "theta_e"), ATHE),
     "PSI": (("z", "theta_e"), psi)},
    coords={"z": z, "theta_e": theta},
    attrs={"source": str(DATA_DIR), "nsnapshots": ntimes},
)
out.to_netcdf("iso_streamfunction.nc")